# Taxonomy Clustering

## Loading (Lib + df)

In [1]:
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))

In [2]:
import pandas as pd
import numpy as np

DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"

In [3]:
USE_INTERSECTION = False  # True  → intersect mistral ∩ deepseek (aligned pair)


def load_and_enrich(name):
    return (
        pd.read_parquet(DATA_DIR / "taxonomy" / "per_llm" / f"{name}.parquet").merge(
            pd.read_parquet(DATA_DIR / "taxonomy" / "per_llm" / "google.parquet")[
                ["bibkey", "google_task_type", "title", "abstract"]
            ],
            on="bibkey",
        )
        # .merge(
        #     pd.read_parquet(
        #         DATA_DIR / "taxonomy" / "per_llm" / "openai_coarse.parquet"
        #     )[["bibkey", "openai_coarse_task"]],
        #     on="bibkey",
        # )
    )


mistral_taxonomy = load_and_enrich("mistral_extraction_full")
print(f"Après load+merge — mistral: {len(mistral_taxonomy):,}")

if USE_INTERSECTION:
    deepseek_taxonomy = load_and_enrich("deepseek_extraction")
    print(f"Après load+merge — deepseek: {len(deepseek_taxonomy):,}")

    common = set(mistral_taxonomy.bibkey) & set(deepseek_taxonomy.bibkey)
    mistral_taxonomy = mistral_taxonomy[mistral_taxonomy.bibkey.isin(common)]
    deepseek_taxonomy = deepseek_taxonomy[deepseek_taxonomy.bibkey.isin(common)]
    print(f"Après intersection — {len(common):,} bibkeys")

    for df in (mistral_taxonomy, deepseek_taxonomy):
        df["is_multi_record"] = df.groupby("bibkey")["bibkey"].transform("size") > 1

    single = set(
        mistral_taxonomy.loc[~mistral_taxonomy.is_multi_record, "bibkey"]
    ) & set(deepseek_taxonomy.loc[~deepseek_taxonomy.is_multi_record, "bibkey"])
    mistral_taxonomy = mistral_taxonomy[mistral_taxonomy.bibkey.isin(single)]
    deepseek_taxonomy = deepseek_taxonomy[deepseek_taxonomy.bibkey.isin(single)]
    print(f"Après filtre multi — {len(single):,} bibkeys")

    mistral_taxonomy = mistral_taxonomy.sort_values("bibkey").reset_index(drop=True)
    deepseek_taxonomy = deepseek_taxonomy.sort_values("bibkey").reset_index(drop=True)
    assert (mistral_taxonomy.bibkey.values == deepseek_taxonomy.bibkey.values).all()
    print("Ordre bibkeys aligné ✓")

else:
    mistral_taxonomy["is_multi_record"] = (
        mistral_taxonomy.groupby("bibkey")["bibkey"].transform("size") > 1
    )
    mistral_taxonomy = mistral_taxonomy[~mistral_taxonomy.is_multi_record]
    mistral_taxonomy = mistral_taxonomy.sort_values("bibkey").reset_index(drop=True)
    print(f"Après filtre multi — {len(mistral_taxonomy):,} bibkeys")
    deepseek_taxonomy = None
    embeddings_deepseek = None


Après load+merge — mistral: 2,305
Après filtre multi — 1,595 bibkeys


In [4]:
if USE_INTERSECTION:
    embeddings_mistral = np.load(
        DATA_DIR / "taxonomy" / "embeddings" / "mistral_embeddings_300.npy"
    )
    embeddings_deepseek = np.load(
        DATA_DIR / "taxonomy" / "embeddings" / "deepseek_embeddings_300.npy"
    )
else:
    embeddings_mistral = np.load(
        DATA_DIR / "taxonomy" / "embeddings" / "mistral_embeddings.npy"
    )
    embeddings_mistral_umap = np.load(
        DATA_DIR / "taxonomy" / "per_llm" / "mistral_embeddings_umap.npy"
    )

print(f"Embeddings chargés — mistral: {embeddings_mistral.shape}")

Embeddings chargés — mistral: (1595, 768)


## 1. Embeddings

In [ ]:
from src.taxonomy.embedding import compute_embeddings

embeddings_mistral = compute_embeddings(
    mistral_taxonomy["paraphrase_task"].astype(str).tolist()
)
np.save(
    DATA_DIR / "taxonomy" / "embeddings" / "mistral_embeddings.npy", embeddings_mistral
)

# embeddings_deepseek = compute_embeddings(
#     deepseek_taxonomy["paraphrase_task"].astype(str).tolist()
# )
# np.save(
#     DATA_DIR / "taxonomy" / "per_llm" / "deepseek_embeddings_300.npy",
#     embeddings_deepseek,
# )

In [ ]:
from src.taxonomy.embedding import compute_embeddings

# enlever les paraphrase domain vide
mistral_taxonomy = mistral_taxonomy[mistral_taxonomy["paraphrase_domain"].notna()]

embeddings_mistral = compute_embeddings(
    mistral_taxonomy["paraphrase_domain"].astype(str).tolist()
)
np.save(
    DATA_DIR / "taxonomy" / "embeddings" / "mistral_embeddings_domain.npy",
    embeddings_mistral,
)

In [ ]:
from src.taxonomy.embedding import compute_embeddings_instruct

embeddings_mistral_instruct = compute_embeddings_instruct(
    mistral_taxonomy["paraphrase_task"].astype(str).tolist()
)
np.save(
    DATA_DIR / "taxonomy" / "per_llm" / "mistral_embeddings_instruct_300.npy",
    embeddings_mistral_instruct,
)


In [ ]:
from src.taxonomy.embedding import umap_projection

embeddings_mistral_umap = umap_projection(embeddings_mistral, n_components=10)

In [ ]:
np.save(
    DATA_DIR / "taxonomy" / "per_llm" / "mistral_embeddings_umap.npy",
    embeddings_mistral_umap,
)

## 2. HAC Construction

In [5]:
from src.taxonomy.hac_utils import (
    build_linkage,
    make_color_map,
    plot_dendrogram,
)

# ── Linkage matrices ──────────────────────────────────────────────────────────
embeddings_arr_m = np.array(embeddings_mistral)
# embeddings_arr_d = np.array(embeddings_deepseek)

embeddings_arr_m_umap = np.array(embeddings_mistral_umap)

Z_m = build_linkage(embeddings_arr_m)
# Z_d = build_linkage(embeddings_arr_d)

Z_m_umap = build_linkage(embeddings_arr_m_umap)

In [6]:
# Palette partagée (mêmes couleurs pour les deux dendrogrammes)
coarse_values_m = mistral_taxonomy["domain"].fillna("Unknown")
# coarse_values_d = deepseek_taxonomy["domain"].fillna("Unknown")

color_map = make_color_map(
    pd.concat(
        [
            coarse_values_m,
            # coarse_values_d
        ]
    )
)

In [ ]:
# Dendrogramme Mistral
fig_m = plot_dendrogram(
    embeddings_arr_m_umap,
    mistral_taxonomy,
    coarse_col="judgment_type",
    label_col="judgment_type",
    title="HAC — Mistral (coloré par coarse label)",
    color_map=color_map,
)
fig_m.show(
    config={"responsive": True, "scrollZoom": True, "displayModeBar": True},
    renderer="browser",
)

In [ ]:
# # Dendrogramme DeepSeek
# fig_d = plot_dendrogram(
#     embeddings_arr_d,
#     deepseek_taxonomy,
#     coarse_col="openai_coarse_task",
#     label_col="openai_coarse_task",
#     title="HAC — DeepSeek (coloré par coarse label)",
#     color_map=color_map,
# )
# fig_d.show(config={"responsive": True, "scrollZoom": True, "displayModeBar": True})


## 3. Clustering HAC Cutting

In [7]:
from src.taxonomy.hac_utils import (
    cut_tree_adaptive,
)

# -- Adaptive cut on Mistral embeddings --
taxo_tree_m = cut_tree_adaptive(Z_m, embeddings_arr_m, min_gain=0.015)
taxo_tree_m_umap = cut_tree_adaptive(Z_m_umap, embeddings_arr_m, min_gain=0.015)

leaf_nodes = taxo_tree_m._leaf_nodes()
print(f"Clusters: {len(leaf_nodes)}")
print(f"Sizes: {sorted([len(n.leaves) for n in leaf_nodes], reverse=True)}")

leaf_nodes_umap = taxo_tree_m_umap._leaf_nodes()
print(f"Clusters (UMAP): {len(leaf_nodes_umap)}")
print(f"Sizes (UMAP): {sorted([len(n.leaves) for n in leaf_nodes_umap], reverse=True)}")


Clusters: 198
Sizes: [50, 50, 47, 45, 44, 42, 40, 36, 36, 36, 29, 25, 23, 22, 21, 21, 21, 20, 19, 18, 18, 18, 17, 15, 15, 15, 15, 14, 14, 14, 14, 13, 12, 12, 11, 11, 11, 10, 10, 10, 10, 9, 9, 9, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Clusters (UMAP): 164
Sizes (UMAP): [46, 44, 43, 41, 40, 39, 38, 37, 33, 30, 30, 28, 25, 23, 22, 20, 19, 19, 19, 18, 18, 18, 18, 18, 17, 17, 16, 16, 15, 15, 15, 15, 14, 14, 13, 13, 13, 12, 12, 12, 12, 11, 11, 11, 10, 10, 10, 10, 9, 9, 9, 9, 9, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6

In [8]:
# Attach cluster assignments to mistral_taxonomy and save
flat = taxo_tree_m_umap.flat_labels(n=len(mistral_taxonomy))

leaf_nodes = taxo_tree_m_umap._leaf_nodes()
# cluster_label_map = {
#     cid: df_row["openai_coarse_task"].value_counts().index[0]
#     for cid, node in enumerate(leaf_nodes)
#     for df_row in [mistral_taxonomy.reset_index(drop=True).iloc[node.leaves]]
# }

mistral_taxonomy = mistral_taxonomy.reset_index(drop=True).copy()
mistral_taxonomy["hac_cluster_id"] = flat
# mistral_taxonomy["hac_cluster_label"] = mistral_taxonomy["hac_cluster_id"].map(
#     cluster_label_map
# )


clustering_df = mistral_taxonomy[
    [
        "bibkey",
        # "paraphrase_task",
        # "google_task_type",
        # "judgment_type",
        "paraphrase_domain",
        "domain",
        "hac_cluster_id",
        # "hac_cluster_label",
        "title",
        "abstract",
    ]
]

mistral_taxonomy[
    [
        "hac_cluster_id",
        # "hac_cluster_label"
    ]
].value_counts()

hac_cluster_id
146               46
9                 44
140               43
76                41
124               40
                  ..
64                 3
134                3
92                 3
56                 3
77                 1
Name: count, Length: 164, dtype: int64

In [9]:
import plotly.colors as pc

df_plot = mistral_taxonomy.copy()
df_plot["hac_cluster_id_str"] = df_plot["hac_cluster_id"].astype(str)

n_clusters = df_plot["hac_cluster_id"].nunique()
palette = pc.qualitative.Alphabet if n_clusters > 24 else pc.qualitative.Plotly
color_map_hac = make_color_map(df_plot["hac_cluster_id_str"], palette=palette)

fig_hac = plot_dendrogram(
    embeddings_arr_m_umap,
    df_plot,
    coarse_col="hac_cluster_id_str",
    label_col="hac_cluster_id_str",
    title=f"HAC — coloré par cluster ({n_clusters} clusters)",
    color_map=color_map_hac,
)
fig_hac.show(
    config={"responsive": True, "scrollZoom": True, "displayModeBar": True},
    renderer="browser",
)


In [ ]:
import json

HAC_DIR = DATA_DIR / "taxonomy" / "hac_clustering"
HAC_DIR.mkdir(parents=True, exist_ok=True)

# Subset historiquement consommé en aval (gardé tel quel pour compat).
# clustering_df.to_parquet(HAC_DIR / "clustering_hac_mistral.parquet", index=False)

# Artefacts complets pour le notebook 5 (LLM merge) — pas de recompute requis.
mistral_taxonomy.to_parquet(HAC_DIR / "clustering_task_full.parquet", index=False)
np.save(HAC_DIR / "embeddings_task_full.npy", embeddings_arr_m)
with (HAC_DIR / "taxo_tree_task.json").open("w", encoding="utf-8") as f:
    json.dump(taxo_tree_m_umap.to_dict(), f, ensure_ascii=False, indent=2)
print(f"Saved clustering artefacts to {HAC_DIR}")

In [12]:
from sklearn.metrics.pairwise import cosine_distances
import json


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


# ── Centroïdes par cluster ────────────────────────────────────────────────────
cluster_ids = sorted(mistral_taxonomy["hac_cluster_id"].unique())
centroids = np.array(
    [
        embeddings_arr_m[mistral_taxonomy["hac_cluster_id"] == cid].mean(axis=0)
        for cid in cluster_ids
    ]
)

# ── Distance inter-cluster (cosine) ──────────────────────────────────────────
dist_matrix = cosine_distances(centroids)
dist_df = pd.DataFrame(dist_matrix, index=cluster_ids, columns=cluster_ids)

# ── Résumé par cluster (avec papiers) ────────────────────────────────────────
cluster_summary = (
    mistral_taxonomy.groupby("hac_cluster_id")
    .agg(
        n_papers=("bibkey", "count"),
        bibkeys=("bibkey", list),
        titles=("title", list),
        domains=("domain", lambda x: x.dropna().value_counts().to_dict()),
    )
    .reset_index()
)
cluster_summary["centroid_idx"] = cluster_summary["hac_cluster_id"].apply(
    lambda cid: cluster_ids.index(cid)
)

# ── Sauvegarde ────────────────────────────────────────────────────────────────
HAC_DIR = DATA_DIR / "taxonomy" / "hac_clustering"

dist_df.to_csv(HAC_DIR / "inter_cluster_distances.csv")

# Pour le parquet : sérialise les listes en JSON
cluster_summary_flat = cluster_summary.copy()
for col in ["bibkeys", "titles", "domains"]:
    cluster_summary_flat[col] = cluster_summary_flat[col].apply(
        lambda x: json.dumps(x, cls=NumpyEncoder)
    )
cluster_summary_flat.drop(columns="centroid_idx").to_parquet(
    HAC_DIR / "cluster_summary.parquet", index=False
)

print(f"inter_cluster_distances.csv — {len(cluster_ids)}×{len(cluster_ids)} clusters")
print(
    f"cluster_summary.parquet    — {len(cluster_summary)} clusters, {mistral_taxonomy['bibkey'].nunique()} papiers"
)
print()
print(dist_df.round(3).to_string())

inter_cluster_distances.csv — 164×164 clusters
cluster_summary.parquet    — 164 clusters, 1595 papiers

       0      1      2      3      4      5      6      7      8      9      10     11     12     13     14     15     16     17     18     19     20     21     22     23     24     25     26     27     28     29     30     31     32     33     34     35     36     37     38     39     40     41     42     43     44     45     46     47     48     49     50     51     52     53     54     55     56     57     58     59     60     61     62     63     64     65     66     67     68     69     70     71     72     73     74     75     76     77     78     79     80     81     82     83     84     85     86     87     88     89     90     91     92     93     94     95     96     97     98     99     100    101    102    103    104    105    106    107    108    109    110    111    112    113    114    115    116    117    118    119    120    121    122    123    124    125    126    